# Retail Sales Analysis & Business Intelligence Workflow
### End-to-End Portfolio Project for Data Analyst & BI Roles

**Author:** Senior Data Analyst & Python Developer  
**Dataset:** Apex Retail India (Simulated Realistic Transaction Data)  
**Currency:** Indian Rupee (INR - ₹)  

---

## 1. Project Overview
This notebook provides an interactive walkthrough of the complete analytical pipeline:
1. **Raw Data Ingestion & Quality Diagnostics**
2. **Data Cleaning & Anomaly Resolution**
3. **Exploratory Data Analysis (EDA) & Key Retail Metrics**
4. **Visual Storytelling & Chart Generation**
5. **SQLite Database Loading & SQL Query Verification**

In [ ]:
# Environment Setup & Library Imports
import os
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Configure plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 10

# Define relative paths
RAW_DATA_PATH = os.path.join('..', 'data', 'raw', 'retail_sales_raw.csv')
CLEANED_DATA_PATH = os.path.join('..', 'data', 'processed', 'retail_sales_cleaned.csv')
DB_PATH = os.path.join('..', 'data', 'processed', 'retail_sales.db')
print('Setup complete. Ready to ingest data.')

## 2. Ingesting Raw Data & Diagnosing Quality Anomalies
Let's inspect the raw dataset to understand the schema, view missing values, examine inconsistent casing, and identify duplicates.

In [ ]:
df_raw = pd.read_csv(RAW_DATA_PATH)
print(f'Raw dataset shape: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns')
display(df_raw.head())

# Missing values diagnostic
print('\nMissing values per column:')
print(df_raw.isnull().sum())

## 3. Data Cleaning & Validation
We now load the cleaned dataset generated by `src/clean_data.py` to confirm that all records adhere to business validation rules.

In [ ]:
df_clean = pd.read_csv(CLEANED_DATA_PATH)
df_clean['Order Date'] = pd.to_datetime(df_clean['Order Date'])
print(f'Cleaned dataset shape: {df_clean.shape[0]} rows, {df_clean.shape[1]} columns')
print(f'Missing values: {df_clean.isnull().sum().sum()}')
display(df_clean.info())

## 4. Calculating the 15 Core Retail Business Metrics
Let's compute executive KPIs: Total Revenue, Unique Orders, Average Order Value (AOV), and Total Units Sold.

In [ ]:
total_sales = df_clean['Sales Amount'].sum()
unique_orders = df_clean['Order ID'].nunique()
aov = total_sales / unique_orders
total_units = df_clean['Quantity'].sum()
unique_products = df_clean['Product ID'].nunique()

print('='*50)
print('EXECUTIVE BUSINESS KPIS (IN INR)')
print('='*50)
print(f'Total Sales Revenue:      INR {total_sales:,.2f}')
print(f'Total Unique Orders:      {unique_orders:,}')
print(f'Average Order Value (AOV):INR {aov:,.2f}')
print(f'Total Units Sold:         {total_units:,}')
print(f'Active Catalog Products:  {unique_products}')
print('='*50)

### Product-Level Analysis: Top 5 by Revenue and Quantity

In [ ]:
prod_summary = df_clean.groupby(['Product ID', 'Product Name', 'Category']).agg(
    Units_Sold=('Quantity', 'sum'),
    Total_Revenue=('Sales Amount', 'sum')
).reset_index()

top_5_rev = prod_summary.sort_values(by='Total_Revenue', ascending=False).head(5)
top_5_qty = prod_summary.sort_values(by='Units_Sold', ascending=False).head(5)

print('TOP 5 PRODUCTS BY REVENUE:')
display(top_5_rev[['Product Name', 'Category', 'Units_Sold', 'Total_Revenue']])

print('\nTOP 5 PRODUCTS BY QUANTITY SOLD:')
display(top_5_qty[['Product Name', 'Category', 'Units_Sold', 'Total_Revenue']])

### Monthly Trend & Month-over-Month (MoM) Growth Rate

In [ ]:
df_clean['YearMonth'] = df_clean['Order Date'].dt.strftime('%Y-%m')
monthly = df_clean.groupby('YearMonth').agg(
    Monthly_Orders=('Order ID', 'nunique'),
    Units_Sold=('Quantity', 'sum'),
    Monthly_Revenue=('Sales Amount', 'sum')
).reset_index()

monthly['MoM_Growth_Pct'] = (monthly['Monthly_Revenue'].pct_change() * 100).round(2)
display(monthly)

### Regional and Category Breakdown

In [ ]:
# Regional performance
reg_perf = df_clean.groupby('Region').agg(
    Orders=('Order ID', 'nunique'),
    Units=('Quantity', 'sum'),
    Revenue=('Sales Amount', 'sum')
).reset_index().sort_values(by='Revenue', ascending=False)
reg_perf['Share_Pct'] = (reg_perf['Revenue'] / total_sales * 100).round(2)

print('REGIONAL PERFORMANCE:')
display(reg_perf)

# Category performance
cat_perf = df_clean.groupby('Category').agg(
    Orders=('Order ID', 'nunique'),
    Units=('Quantity', 'sum'),
    Revenue=('Sales Amount', 'sum')
).reset_index().sort_values(by='Revenue', ascending=False)
cat_perf['Contribution_Pct'] = (cat_perf['Revenue'] / total_sales * 100).round(2)

print('\nCATEGORY PERFORMANCE:')
display(cat_perf)

## 5. Visual Storytelling & Exploratory Plots

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Monthly Trend Plot
ax1.plot(monthly['YearMonth'], monthly['Monthly_Revenue'] / 100000.0, marker='o', color='#0e7490', lw=2.5)
ax1.set_title('Monthly Revenue (INR Lakhs)', fontweight='bold')
ax1.set_ylabel('INR Lakhs')
ax1.set_xticks(range(len(monthly['YearMonth'])))
ax1.set_xticklabels(monthly['YearMonth'], rotation=45, ha='right')
ax1.grid(True, linestyle='--', alpha=0.5)

# Category Contribution Bar Plot
bars = ax2.bar(cat_perf['Category'], cat_perf['Revenue'] / 100000.0, color='#4338ca', width=0.55)
ax2.set_title('Revenue by Product Category (INR Lakhs)', fontweight='bold')
ax2.tick_params(axis='x', rotation=25)
ax2.grid(True, linestyle='--', alpha=0.5)
for bar in bars:
    h = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width() / 2, h + 0.1, f'{h:.1f}L', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

## 6. SQL Query Execution in SQLite
We now connect to `data/processed/retail_sales.db` and execute SQL queries to demonstrate that our SQL queries match our Python metrics.

In [ ]:
conn = sqlite3.connect(DB_PATH)
sql_query = '''
SELECT 
    ROUND(SUM(sales_amount), 2) AS total_revenue_inr,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(sales_amount) * 1.0 / COUNT(DISTINCT order_id), 2) AS average_order_value_inr,
    SUM(quantity) AS total_quantity_sold
FROM retail_sales;
'''
df_sql_kpis = pd.read_sql_query(sql_query, conn)
print('SQL Query Output:')
display(df_sql_kpis)
conn.close()

## 7. Conclusions & Next Steps
- All 15 required business metrics have been validated across Python and SQL.
- Proceed to `powerbi/powerbi_dashboard_guide.md` to build the interactive Power BI report.
- Proceed to `excel/excel_analysis_guide.md` to inspect PivotTables and formulas.